# UdaPlay — Notebook 02: Agent Implementation

This notebook demonstrates the full **UdaPlay AI Research Agent** end-to-end:

- **Tool 1** `retrieve_game` — semantic search over the ChromaDB knowledge base
- **Tool 2** `evaluate_retrieval` — Claude scores how well results answer the query
- **Tool 3** `game_web_search` — Tavily web fallback when confidence is low
- **State machine** — `IDLE → RETRIEVE → EVALUATE → [ANSWER | WEBSEARCH → PERSIST] → ANSWER`
- **Long-term memory** — web results persisted to ChromaDB for future queries
- **Multi-turn sessions** with conversation summary

> **Prerequisite:** Run Notebooks 00 and 01 first to populate and index the game dataset.

## 1. Setup

In [ ]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from dotenv import load_dotenv
load_dotenv(os.path.join(project_root, '.env'))

import anthropic
from tavily import TavilyClient
import chromadb

CHROMA_PERSIST_DIR = os.path.join(project_root, os.getenv('CHROMA_PERSIST_DIR', 'data/chroma_db'))

llm_client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
tavily_client = TavilyClient(api_key=os.getenv('TAVILY_API_KEY'))
chroma_client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)

print('✓ Clients initialized')
print(f'  ChromaDB path: {CHROMA_PERSIST_DIR}')

In [ ]:
from src.embeddings import EmbeddingManager
from src.vector_store import VectorStoreManager

embedding_mgr = EmbeddingManager(backend='sentence-transformers')
vector_store = VectorStoreManager(
    chroma_client=chroma_client,
    embedding_manager=embedding_mgr,
)

stats = vector_store.get_collection_stats()
print(f'✓ Vector store ready — {stats["count"]} games indexed')
assert stats['count'] > 0, 'Run Notebook 01 first to index games!'

## 2. Tool 1 — `retrieve_game`

Semantic search over ChromaDB. Returns ranked `RetrievalResult` objects.

In [ ]:
import pandas as pd
from IPython.display import display
from src.tools import retrieve_game

def show_retrieval(query: str, n: int = 5):
    results = retrieve_game(query, vector_store, n_results=n)
    df = pd.DataFrame([{
        'Rank': i+1,
        'Title': r.title,
        'Relevance': f'{r.relevance_score:.3f}',
        'Developer': r.metadata.get('developer', ''),
        'Genre': r.metadata.get('genre', ''),
        'Source': r.metadata.get('source', ''),
    } for i, r in enumerate(results)])
    print(f'\n🔍 Query: "{query}" ({len(results)} results)')
    display(df.style.background_gradient(subset=['Relevance'], cmap='Greens'))
    return results

results = show_retrieval('open world action RPG game', n=5)

In [ ]:
# Show the top result's full document
if results:
    top = results[0]
    print(f'Top result: {top.title}')
    print(f'Distance:   {top.distance:.4f}')
    print(f'Relevance:  {top.relevance_score:.3f}')
    print(f'\nEmbedded document:')
    print(top.document)

## 3. Tool 2 — `evaluate_retrieval`

Claude scores how well retrieved documents answer the query.  Returns `EvaluationResult` with `confidence` and `should_web_search`.

In [ ]:
from src.tools import evaluate_retrieval

def show_evaluation(query: str):
    results = retrieve_game(query, vector_store, n_results=5)
    eval_result = evaluate_retrieval(query, results, llm_client, threshold=0.65)
    
    badge = '✓ Use RAG answer' if not eval_result.should_web_search else '⚠ Trigger web search'
    print(f'Query:      "{query}"')
    print(f'Confidence: {eval_result.confidence:.2f}  →  {badge}')
    print(f'Reason:     {eval_result.reason}')
    print(f'Relevant:   {[r.title for r in eval_result.sufficient_results]}')
    return eval_result

In [ ]:
# High confidence — game is in our knowledge base
eval_high = show_evaluation('Who developed Hollow Knight?')

In [ ]:
# Low confidence — game likely not in our knowledge base → web search needed
eval_low = show_evaluation('What games won Game of the Year in 2025?')

## 4. Tool 3 — `game_web_search`

Tavily API search, automatically prefixed with `"video game:"` to focus results.

In [ ]:
from src.tools import game_web_search

web_results = game_web_search('Indiana Jones and the Great Circle game 2024', tavily_client, max_results=3)

print(f'Web search returned {len(web_results)} results:\n')
for i, r in enumerate(web_results):
    print(f'[{i+1}] {r.title}')
    print(f'    Score: {r.score:.3f}')
    print(f'    URL:   {r.url}')
    print(f'    {r.content[:150]}...')
    print()

## 5. Long-Term Memory — Persist Web Results

Web search results are normalized into the game schema and upserted into ChromaDB.  Future queries can then retrieve them from the internal knowledge base.

In [ ]:
from src.agent import _web_result_to_game_doc

count_before = vector_store.get_collection_stats()['count']
print(f'Collection count before: {count_before}')

if web_results:
    docs = [_web_result_to_game_doc(r, 'Indiana Jones game') for r in web_results[:1]]
    vector_store.upsert_documents(docs)
    count_after = vector_store.get_collection_stats()['count']
    print(f'Collection count after:  {count_after}')
    print(f'\n✓ {count_after - count_before} new entry/entries persisted to ChromaDB')
    print(f'  source="web_search" entries are now retrievable in future queries')

## 6. Agent State Machine Walkthrough

Tracing through each state handler manually to show the state transitions.

In [ ]:
from src.agent import UdaPlayAgent, AgentContext, AgentState

agent = UdaPlayAgent(
    vector_store=vector_store,
    llm_client=llm_client,
    tavily_client=tavily_client,
    confidence_threshold=0.65,
    answer_format='both',
)

print('UdaPlayAgent initialized')
print(f'Confidence threshold: {agent._threshold}')
print(f'Default output format: {agent._default_format}')

In [ ]:
# Manually step through the state machine for a single query
query = 'Who published Elden Ring?'
ctx = AgentContext(query=query, session_id='demo-session')
print(f'Initial state: {ctx.state.value}')

# Step 1: IDLE → RETRIEVE
ctx = agent._handle_retrieve(ctx)
print(f'After retrieve: {ctx.state.value} — {len(ctx.retrieval_results)} results')

# Step 2: RETRIEVE → EVALUATE
ctx = agent._handle_evaluate(ctx)
print(f'After evaluate: {ctx.state.value} — confidence={ctx.confidence_score:.2f}')

# Step 3: EVALUATE → ANSWER (or WEBSEARCH)
if ctx.confidence_score >= agent._threshold:
    print(f'High confidence → skipping web search')
    ctx = agent._handle_answer(ctx)
else:
    print(f'Low confidence → running web search')
    ctx = agent._handle_websearch(ctx)
    ctx = agent._handle_persist(ctx)
    ctx = agent._handle_answer(ctx)

print(f'Final state: {ctx.state.value}')
print(f'Answer preview: {ctx.final_answer[:200]}...')

## 7. Full Agent Runs — Example Queries

Three representative queries demonstrating the full `agent.run()` pipeline.

In [ ]:
from IPython.display import Markdown

def run_query(query: str, session_id: str = None, label: str = None):
    print(f'\n{"="*60}')
    print(f'Query: "{query}"')
    if label:
        print(f'({label})')
    print('='*60)
    
    ctx = agent.run(query, session_id=session_id)
    formatted = agent.get_formatted_answer(ctx)
    
    print(f'State:       {ctx.state.value}')
    print(f'Confidence:  {ctx.confidence_score:.2f}')
    print(f'Web search:  {"Yes" if ctx.web_results else "No"}')
    print(f'Citations:   {len(ctx.citations)}')
    print()
    
    if isinstance(formatted, dict) and 'text' in formatted:
        display(Markdown(formatted['text']))
        print('\nStructured JSON:')
        import json
        print(json.dumps(formatted['json'], indent=2))
    else:
        print(formatted)
    
    return ctx

In [ ]:
# Query 1: Game in knowledge base — should get high confidence RAG answer
ctx1 = run_query(
    'Who developed Hollow Knight and what platforms is it available on?',
    label='Expected: RAG hit, high confidence'
)

In [ ]:
# Query 2: Publisher question — clear factual answer in dataset
ctx2 = run_query(
    'Who published Elden Ring and when was it released?',
    label='Expected: RAG hit, factual answer'
)

In [ ]:
# Query 3: Beyond the dataset — should trigger web search
ctx3 = run_query(
    'What are the most anticipated video games coming out in late 2025?',
    label='Expected: low confidence → web search fallback'
)

## 8. ReportFormatter — Output Modes

In [ ]:
from src.reporter import ReportFormatter
import json

fmt = ReportFormatter()

# Use the answer from Query 1
answer = ctx1.final_answer
citations = ctx1.citations
confidence = ctx1.confidence_score
query = ctx1.query

print('--- TEXT MODE ---')
text_output = fmt.format(answer, citations, mode='text', confidence=confidence, query=query)
display(Markdown(text_output))

In [ ]:
print('--- JSON MODE ---')
json_output = fmt.format(answer, citations, mode='json', confidence=confidence, query=query)
print(json.dumps(json_output, indent=2))

In [ ]:
print('--- BOTH MODE ---')
both_output = fmt.format(answer, citations, mode='both', confidence=confidence, query=query)
print('text:')
display(Markdown(both_output['text']))
print('\njson:')
print(json.dumps(both_output['json'], indent=2))

## 9. Multi-Turn Conversation Session

In [ ]:
SESSION_ID = 'demo-multi-turn'

questions = [
    'Who developed Stardew Valley?',
    'What genre is it and what platforms can I play it on?',
    'How does it compare to other farming games like Harvest Moon?',
]

session_contexts = []
for i, question in enumerate(questions, 1):
    print(f'\n[Turn {i}] {question}')
    ctx = agent.run(question, session_id=SESSION_ID)
    session_contexts.append(ctx)
    print(f'  Confidence: {ctx.confidence_score:.2f} | Web search: {"Yes" if ctx.web_results else "No"}')
    print(f'  Answer preview: {ctx.final_answer[:150]}...')

In [ ]:
# Show session history
history = agent.memory.get_session_history(SESSION_ID)
print(f'Session "{SESSION_ID}" — {len(history)} turns recorded')
print()
for i, ctx in enumerate(history, 1):
    print(f'Turn {i}: {ctx.query}')
    print(f'  → Confidence: {ctx.confidence_score:.2f}, Citations: {len(ctx.citations)}')

In [ ]:
# Get Claude's summary of the session
summary = agent.memory.get_conversation_summary(SESSION_ID, llm_client)
print('Session summary:')
print(summary)

## 10. Performance Metrics

In [ ]:
# Collect all contexts from this notebook
all_contexts = [ctx1, ctx2, ctx3] + session_contexts

rag_hits = [c for c in all_contexts if not c.web_results]
web_hits = [c for c in all_contexts if c.web_results]

final_count = vector_store.get_collection_stats()['count']

print('=== UdaPlay Agent Performance Summary ===')
print(f'Total queries run:        {len(all_contexts)}')
print(f'Answered from RAG:        {len(rag_hits)} ({100*len(rag_hits)//len(all_contexts)}%)')
print(f'Fell back to web search:  {len(web_hits)} ({100*len(web_hits)//len(all_contexts)}%)')
print(f'Avg confidence (RAG):     {sum(c.confidence_score for c in rag_hits)/max(len(rag_hits),1):.2f}')
print(f'Knowledge base size now:  {final_count} documents')

web_search_docs = final_count - sum(1 for g in [] if g.get('source') == 'internal')
print()
print('Knowledge base growth:')
print(f'  web_search entries added this session: {len(web_hits)} (persisted to ChromaDB)')
print(f'  These will be available for future queries without web search')

## Summary

| Feature | Implementation |
|---|---|
| **RAG retrieval** | `retrieve_game()` → ChromaDB semantic search |
| **Quality evaluation** | `evaluate_retrieval()` → Claude confidence scoring |
| **Web fallback** | `game_web_search()` → Tavily API |
| **State machine** | `IDLE→RETRIEVE→EVALUATE→[ANSWER\|WEBSEARCH→PERSIST]→ANSWER` |
| **Long-term memory** | Web results upserted to ChromaDB with `source="web_search"` |
| **Session memory** | `AgentMemory` tracks per-session conversation history |
| **Structured output** | `ReportFormatter` → text, JSON, or both |
| **Multi-turn** | Same `session_id` accumulates context across turns |